# GlucoRisk AI: Subgroup Audit

Explore feature-defined subgroups; this is not a protected-group fairness guarantee.

**Synthetic educational data only.** Each notebook is independently runnable. Validation analyses do not establish clinical effectiveness.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Run from project root or notebooks directory')
sys.path.insert(0,str(ROOT/'src'))
from healthcare_ai.settings import FEATURES,TARGET
from healthcare_ai.io import load_csv
from healthcare_ai.splits import split
from healthcare_ai.pipeline import build
from healthcare_ai.metrics import evaluate
frame=load_csv(ROOT/'data/demo.csv')
train,val,test=split(frame)
model=build().fit(train[FEATURES],train[TARGET])


In [2]:
from healthcare_ai.fairness import subgroup_metrics
p=model.predict_proba(val[FEATURES])[:,1]
groups=np.where(val[FEATURES[0]]>=train[FEATURES[0]].median(),'above_training_median','below_training_median')
print(json.dumps(subgroup_metrics(val[TARGET],p,groups),indent=2))

{
  "above_training_median": {
    "auroc": 0.829861111111111,
    "average_precision": 0.8928312551281424,
    "brier": 0.1685469325065032,
    "balanced_accuracy": 0.75,
    "sensitivity": 0.8333333333333334,
    "specificity": 0.6666666666666666,
    "tp": 30,
    "tn": 16,
    "fp": 8,
    "fn": 6,
    "n": 60,
    "threshold": 0.5
  },
  "below_training_median": {
    "auroc": 0.8519388954171563,
    "average_precision": 0.7987560098173908,
    "brier": 0.16025811882700844,
    "balanced_accuracy": 0.7643948296122209,
    "sensitivity": 0.8260869565217391,
    "specificity": 0.7027027027027027,
    "tp": 19,
    "tn": 26,
    "fp": 11,
    "fn": 4,
    "n": 60,
    "threshold": 0.5
  }
}


## Interpretation

These outputs describe a mathematical simulator. Review sample size, uncertainty and the documented data contract before adapting this analysis. Repeated validation inspection can overfit development choices; use a separate frozen holdout for final assessment.